# Analytics notebook

This notebook addresses the Analytics section of the technical brief. The objective is to explore the Lending Club portfolio, define the modelling target, identify potential (univariate) risk drivers, and draw insights that can inform credit risk decisioning.

The analysis covers four areas:

* Define what constitutes a “bad” outcome based on the observed loan performance data
* Explore how key tradeline and application variables impact the likelihood of bad outcomes
* Assess how historical credit risk policies may have shaped the current portfolio composition
* Provide additional insights that could influence credit risk decisioning and support the subsequent modelling stage

## 1. Set up

This section sets up the notebook environment and project structure required to run the analysis in Colab.

It includes:

* cloning the GitHub repository
* installing the required Python packages
* setting up the project directory structure
* downloading the Lending Club accepted and rejected application datasets
* defining reusable file paths for the raw data, processed data and model outputs

I've abstracted away commonly used code into functions which are installed as packages in silvercat.utils

In [1]:
# clone the repo
# I only need numpy and pandas in this notebook, which are preinstalled in Collab's runtime so no need for further installations

%cd /content
!rm -rf silvercat
!git clone https://github.com/dimvasilev/silvercat.git
%cd /content/silvercat

/content
Cloning into 'silvercat'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 64 (delta 21), reused 50 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 334.13 KiB | 8.79 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/silvercat


In [2]:
# import packages and project helpers

import sys
import importlib
import gc

sys.path.insert(0, "/content/silvercat/src")
importlib.invalidate_caches()

from silvercat.utils import (
    setup,
    add_loan_date_features,
    object_column_summary,
    missingness_by_year,
    distribution_by_year,
    apply_terminal_target,
    bad_rate_summary,
    bin_segments,
    plot_bad_rate_by_year,
    add_model_candidate_features,
    plot_separation_grid,
    pct_mix,
    yearly_summary,
    plot_mix,
    plot_yearly_metric,
    plot_interest_rate_by_grade,
    count_applications_by_year,
    build_application_volume_summary,
    plot_application_volume_and_acceptance,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.2f}".format)


In [3]:
# the setup helper downloads the datasets and sets up the project paths
# we have to run this in every notebook because notebooks don't share a runtime
# I've implemeneted a simple dataclass with predefined paths so its easy to access the correct path for different accets like this path.accepts, path.rejects etc

paths = setup(project_root="/content/silvercat")

Using Colab cache for faster access to the 'lending-club' dataset.
Successfully copied Lending Club files to /content/silvercat/data
Project root: /content/silvercat
Data directory: /content/silvercat/data
Accepted data exists: True
Rejected data exists: True
Data dictionary exists: True


In [ ]:
# read in the full accepts dataset

accepts_full = pd.read_csv(
    paths.accepts,
    compression="gzip",
    low_memory=False
)

In [ ]:
# create helper date columns used for grouping and analysis

accepts_full = add_loan_date_features(accepts_full)


## 2. High-level data exploration

In this section I perform an initial review of the accepted applications dataset. The objective is to understand the structure of the data, identify potential data-quality issues, and form initial hypotheses for the analytics and modelling stages.

The review covers:

* dataset shape and column types
* basic descriptive statistics for numeric and categorical variables
* missingness patterns
* identification of variables that may be suitable for modelling, analytics only, or exclusions as well as key segments used for policy assessment

To manage Colab memory constraints, this initial EDA is performed on a random sample of 10,000 rows. The findings are used to define a smaller set of key variables that can then be loaded and analysed on the full dataset.

In [ ]:
# take a random 10,000-row sample for initial EDA

accepts_sample = (
    accepts_full
    .sample(n=10_000, random_state=42)
    .copy()
)

In [ ]:
# preview the first rows to inspect column structure and example values.

accepts_sample.head()
# accepts_sample[['id', 'term', 'loan_status', 'next_pymnt_d', 'issue_month', 'last_pymnt_month', 'issue_year', 'months_on_book']].head(100)

### Initial observations from the data preview

The accepted applications dataset contains a mix of application form information, loan/product attributes, credit bureau variables, performance outcomes and loan performance/servicing fields. Since we are building a quote stage credit risk model, these groups need to be treated differently.

Key observations:

* `id` and `member_id` appear to be identifiers and should be excluded from modelling. `member_id` is a Lending Club specific identified, which could potentially be used to identify repeat borrowers, but not as a model feature

* `loan_status` is the main candidate field for defining the bad outcome. This will be reviewed separately to decide which statuses should be classified as good, bad or indeterminate

* `issue_d` is the loan origination date. It is useful for vintage analysis, policy trend analysis and development / validation splits, but should not be used directly as a model predictor

* `loan_amnt`, `funded_amnt`, `funded_amnt_inv`, `term`, `installment`, `grade`, `sub_grade` and `int_rate` appear to reflect product, pricing, funding or underwriting strategy. These variables are useful for portfolio and policy analysis, but are not suitable as core predictors in the main quote stage risk model because they are based on market conditions / strategy

* `title` appears to be a title() case of `purpose`. This relationship should be checked before deciding whether one of these fields can be dropped

* Geographic variables such as `zip_code` and `addr_state` are excluded from the main decisioning model due to fairness considerations

* `fico_range_low` and `fico_range_high` appear to provide credit bureau score bands. These can potentially be combined into a single derived FICO feature, for example by taking the midpoint of the range

* `initial_list_status` appears to reflect Lending Club’s investor listing mechanism rather than borrower creditworthiness. It is therefore more appropriate for analytics than for modelling

* Payment, recoveries, collections, hardship and settlement fields are post-origination performance or servicing variables. These are not available at quote stage and should not be considered for modelling

* `last_credit_pull_d` and related credit-pull fields may be relevant for behavioural monitoring, but are not suitable for quote stage modelling because they can reflect events after the original lending decision

* `application_type` may be retained as binary characteristic, subject to volume and bad-rate checks for individual versus joint applications. If joint applications have sufficient volume and materially different risk, this could be considered as a future segmentation candidate

* Joint-application variables should be reviewed separately. They may be relevant only for joint applications and may need specific missing-value treatment or exclusion from the initial baseline model

* `verification_status` may reflect operational or underwriting workflow rather than pure borrower risk. It will be reviewed carefully before deciding whether it is suitable for modelling

* `policy_code` may provide a limited direct policy flag. Its distribution over time will be reviewed to assess whether it offers any useful insight into historical policy differences

In [ ]:
# review descriptive statistics for numeric variables

accepts_sample.describe().T

### Observations from descriptive statistics for numeric variables

The `describe()` output provides an initial view of the numeric variables, including their coverage, central tendency, spread and extreme values. This helps identify data-quality checks and transformations needed before separation analysis or modelling.

Key observations:

* `member_id` is fully missing in the sample, confirming that it cannot be used for analysis or modelling in its current form

* `loan_amnt`, `funded_amnt` and `funded_amnt_inv` are very similar in distribution, suggesting that requested and funded amounts are usually close. This supports using these fields for portfolio exposure analysis, while still excluding them from the core risk model because they may reflect product or funding strategy

* `annual_inc`, `dti`, `revol_bal`, `tot_cur_bal` and several balance/limit variables show wide ranges and high maximum values. These variables may require outlier treatment, flooring/capping (i.e. windsorisation), transformation or robust binning before modelling. Tree-based models are generally less sensitive to extreme values than linear models, but outliers should still be reviewed for data quality, interpretability and stability

* `dti` has a small number of missing values and a very high maximum value, suggesting possible outliers or unusual cases. This should be reviewed before using the variable in separation analysis or modelling

* `fico_range_low` and `fico_range_high` have identical spread and differ by approximately four points. This supports combining into a single feature using the average / median

* Several delinquency and public-record variables are highly positively skewed, including `delinq_2yrs`, `pub_rec`, `collections_12_mths_ex_med`, `chargeoff_within_12_mths`, `acc_now_delinq`, `tax_liens` and `pub_rec_bankruptcies`. These may be more useful as binary flags or capped rather than used as raw continuous inputs

* `policy_code` is constant at `1` in the sample. This suggests it is unlikely to provide useful segmentation or modelling value, although the distribution should still be checked on the full dataset and over time

* Joint-application variables such as `annual_inc_joint`, `dti_joint`, `revol_bal_joint` and secondary-applicant bureau fields are populated only for a small subset of records. These variables should not be used directly in the baseline model without specific joint-application handling

* Several recently addedd bureau variables, such as `open_acc_6m`, `open_act_il`, `open_il_12m`, `all_util`, `inq_fi` and `inq_last_12m`, have materially lower counts than the core bureau variables. This may indicate that they were introduced later in the dataset history, so their availability should be checked by origination year before inclusion

* `last_fico_range_low` and `last_fico_range_high` are populated but represent updated bureau information after the original decision point. They should be excluded from the origination model

* Same applies for hardship and settlement fields which should not be considered for modelling

Overall, the descriptive stats for numeric variables support the use of quote stage bureau variables as input for core modelling and key segment variables (loan term, amount, apr and average fico score) as input for strategy analysis. I keep these variables alongside the target which is discussed in the next section. The remaining variables are dropped from the dataset.

In [ ]:
# summarise object columns with missingness and cardinality

object_column_summary(accepts_sample)


### Additional observations from string and categorical variables

Next, I perform a similar review of the string variables.

- The `id` is unique confimning that we have 1 row per loan. This means that we don't have a full payment history which in turn influences how we define the outcome target in the next section
- High-cardinality fields such as `url`, `emp_title`, `zip_code`, `desc` and `title` are free-text fields, which are noisy and not suitable for the baseline model without additional text processing. These are excluded
- Several date variables are stored as strings, including `issue_d`, `earliest_cr_line`, `last_pymnt_d`, `last_credit_pull_d`, `next_pymnt_d` and settlement or hardship dates. These need to be converted for usage as temporal features (or as intermediate values in calculations). For example, issue_d appearsto be the loan disbursal date
- Categorical variables such as `loan_status`, `term`, `grade`, `sub_grade`, `purpose`, `verification_status`, `initial_list_status` are useful for portfolio analysis, but not modelling

Overall, this review supports excluding identifiers, free-text fields and loan performance-related variables from the baseline model.

In [ ]:
# % missing values per variable / year

missing_by_year_all = missingness_by_year(accepts_sample)

missing_by_year_all

### Observations from missingness over time

Next, I examine the patterns of missing data by variable over time.

It is evident that missingness does not always appear to be random. For several fields, missingness changes sharply by origination year, suggesting that some variables may have been introduced later in the Lending Club data history or only became consistently populated after a certain point.

Key observations:

* Core application, product and bureau variables such as `loan_amnt`, `term`, `int_rate`, `grade`, `annual_inc`, `loan_status`, `dti`, `fico_range_low`, `fico_range_high`, `open_acc`, `revol_bal` and `total_acc` have strong coverage across the full observation period. This makes them useful for longitudinal portfolio and policy analysis

* The `desc` field becomes increasingly unavailable over time and is fully missing from 2015 onwards. This supports excluding it from further analysis and modelling

* Several bureau variables appear to have been introduced part-way through the dataset. For example, variables such as `tot_coll_amt`, `tot_cur_bal`, `total_rev_hi_lim`, `avg_cur_bal`, `num_*` fields and several trade-line measures are fully missing in earlier years but become populated from around 2012/2013 onwards

* A second group of mroe recent bureau variables, including `open_acc_6m`, `open_act_il`, `open_il_12m`, `open_il_24m`, `all_util`, `inq_fi` and `inq_last_12m`, are almost entirely missing before 2016 but have strong coverage from 2016 onwards. These variables may be useful if we decide to develop the model using development sample of 2016 or later

* Joint-application variables are mostly missing until later years and remain highly sparse even in 2017–2018. This reflects the limited availability of joint-application data and supports excluding the secondary-applicant block from the initial baseline model

* Hardship and settlement variables are almost entirely missing across all years. As discussed, these will be excluded from future analysis and modelling

**Modelling implication:** The analysis so far suggests that we should prioritise stable bureau-derived variables with broad and stable coverage across the full data set or at least after 2015. Bureau variables introduced post 2017 will be excluded from the baseline full-period model to avoid time-period bias, but could be tested separately in a recent-vintage sensitivity model.

## 3. Defining what constitutes a "bad" outcome

This section defines the target variable for the credit risk analysis and subsequent modelling stage. The objective is to translate the observed Lending Club loan performance status into a binary outcome that distinguishes bad loans from good.

The definition of a bad outcome is a key modelling decision. It affects the observed bad rate, the development sample, model performance and the interpretation of the final risk scores. For this reason, the target definition is reviewed explicitly before any separation analysis or modelling is carried out.

I took the following approach:

* review the distribution of `loan_status` values in the accepted applications dataset
* examine how loan statuses vary by origination year to understand the impact of loan maturity and observation window
* classify statuses into good, bad or indeterminate outcome groups
* exclude loans where the final outcome is not yet sufficiently observed, i.e. `indeterminate`
* create a binary `bad` target variable for later analysis and modelling
* review the resulting bad rate over time and across key portfolio segments to inform the modelling development period

After defining the target, I also review the bad-rate distribution by origination year and by key portfolio dimensions such as term, loan amount and interest rate band. This helps assess whether the observed target is stable across vintages and whether the selected modelling period is likely to produce a representative development sample.

From this point onwards, I consider the the full observations accepted applications dataset. To manage memory, only the columns required for target construction and segment diagnostics are loaded.

Data scope:

Section 3:
accepts_target = all rows, key segment columns only

Section 4:
accepts_separation = all rows, model candidate columns only

Section 5:
accepts_policy = all rows, key segment columns only

In [ ]:
# in this section, we only need a subset of all the columns in the dataset

target_cols = [
    "loan_status",
    "issue_d",
    "last_pymnt_d",
    "loan_amnt",
    "term",
    "int_rate",
    "fico_range_low",
    "fico_range_high",
    "issue_year"
]

accepts_target = accepts_full[target_cols].copy()

loan status distribution over time

In [ ]:
# counts

distribution_by_year(
    accepts_target,
    segment_col="loan_status",
    percent=False
)

In [ ]:
# percentages

distribution_by_year(
    accepts_target,
    segment_col="loan_status",
    percent=True
)

### Observations from loan status distribution over time

The `loan_status` distribution changes materially by origination year. Earlier vintages are mostly resolved, with loans mainly classified as `Fully Paid` or `Charged Off`. More recent vintages, especially 2016–2018, contain a much larger share of `Current` loans.

`Current` means the loan is active and up to date at the dataset snapshot. It does not mean the loan has reached a final good outcome. Treating all current loans as good would therefore understate the bad rate for recent vintages.

To account for that I have decided to use a terminal-status target for the baseline model. Loans are classified as bad where they have reached `Charged Off` or `Default`, and good where they have reached `Fully Paid`. Open or unresolved statuses such as `Current`, `Late` and `In Grace Period` are excluded from the baseline target because their final outcome is not yet known.

This prioritises target quality over sample size. Although it does not estimate a specific “bad within 12 months” or “bad within 60 months” outcome, it provides a clean and defensible binary outcome from the information available in the dataset.

Formal target definition:
- Classify Charged Off and Default as bad
- Classify Fully Paid as good
- Current, Late, In Grace Period as indeterminate
- Does not meet credit policy as indeterminate although this status is not observed after 2010

Indetermine observations are excluded from the dataset for the purposes of Analysis and Modelling.

In [ ]:
# define target status groups

bad_statuses = [
    "Charged Off",
    "Default",
]

good_statuses = [
    "Fully Paid",
]

indeterminate_statuses = [
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)",
    "Does not meet the credit policy. Status:Fully Paid",
    "Does not meet the credit policy. Status:Charged Off",
    "Current",
]

In [ ]:
# create binary target flag

accepts_target = apply_terminal_target(accepts_target)


In [ ]:
# bad rate distribution over time

bad_rate_by_year = bad_rate_summary(accepts_target)

bad_rate_by_year

### Observations from bad rate over time

The observed bad rate increases from around 2010 onwards, reaching above 20% for the 2015–2017 vintages. However, this trend should be interpreted cautiously because the target is based on terminal loan statuses only.

The later vintages, particularly 2017 and 2018, are not fully seasoned at the April 2019 snapshot date. Since `Current` loans are excluded from the terminal-status target, these recent years contain only loans that have already reached a final outcome, such as early full repayment or early charge-off. Their bad rates are therefore subject to seasoning bias and should not be treated as final vintage performance.

For this reason, the bad-rate trend is more reliable for older vintages where a larger share of loans has reached a terminal outcome. Recent vintages are retained for exploratory analysis, but for model development I will exclude these vintages.

bin continuous variables

In [ ]:
# create target diagnostic bands

accepts_target = bin_segments(accepts_target)

In [ ]:
# define list of key segments

segments = [
    "term",
    "loan_amount_band",
    "int_rate_band",
    "fico_band",
]

distribution by key segment

In [ ]:
# bad rate distribution over time for key segments

plot_bad_rate_by_year(
    accepts_target,
    segment_cols=segments,
)

### Observations from target distribution by key segment over time

The observed trends by key segment reinforces mhy view that the most appropriate period for model development is 2014–2016.

Across the key segments, the expected risk ordering is generally sensible:
- 60-month loans have materially higher bad rates than 36-month loans
- higher loan amount bands tend to have higher bad rates
- interest-rate bands and fico scores also show a strict monotonic relationship with bad rate

This supports the quality of the target definition and confirms that the selected outcome is capturing meaningful credit risk signal.

However, the overall time trend also shows clear vintage effects. Bad rates increase materially from 2014 to 2016, before falling sharply in 2018. The fall in 2018 should not be interpreted as an improvement in risk quality, because the terminal-status target excludes `Current` loans and the 2017–2018 vintages are not fully seasoned at the April 2019 snapshot. These later years are therefore biased towards loans that reached an early final outcome, such as early full repayment or early charge-off.

The 2007–2013 vintages are also less suitable for main model development because they are older, lower-volume in the early years, and may reflect different underwriting policies, product mix, borrower behaviour and data availability.

For these reasons, I use 2014–2015 for model development and validation, and hold out 2016 as an out-of-time test sample. This provides a recent, high-volume, sufficiently seasoned modelling period while avoiding both older legacy vintages and the most immature recent vintages.

### Final development window selection

- Train / validation with cross-validation: Jan 2014 – Jun 2015
- Test: Jul 2015 – Dec 2015
- Out-of-time: Jan 2016 – Jun 2016

## 4. Explore how key variables impact the likelihood of bad outcomes

This section uses univariate separation analysis to review which quote-stage application and bureau variables show a clear relationship with the target.

The analysis is restricted to the selected development window and includes variables that would be available at the point of application. This includes borrower characteristics such as `home_ownership` and `purpose`, together with bureau variables covering credit history, utilisation, delinquencies, public records, inquiries, balances, limits and account mix.

Variables with insufficient coverage in the development window have been removed from this stage. Product, pricing and underwriting outcome variables such as `loan_amnt`, `term`, `int_rate`, `grade`, `sub_grade` and `installment` are excluded from the modelling feature set. Similarly, and as highlighted above, loan performance related variables are excluded.

Two simple transformations are applied before the analysis: `earliest_cr_line` is converted into account age at origination, and `fico_range_low` / `fico_range_high` are combined into a single average FICO measure.

For each candidate variable, I compare bad rates across simple bands or categories. The purpose is to identify useful risk drivers for the modelling stage, rather than make final feature-selection decisions.

In [ ]:
# filter to development window: Jan 2014 to Jun 2016

accepts_univariate_analysis = accepts_full[
    accepts_full["issue_month"].between("2014-01-01", "2016-06-30")
].copy()

In [ ]:
# recreate the target

accepts_univariate_analysis = apply_terminal_target(accepts_univariate_analysis)

In [ ]:
# candidate quote-stage variables

candidate_vars = [
    "home_ownership", "purpose",
    "delinq_2yrs", "earliest_cr_line",
    "fico_range_low", "fico_range_high",
    "inq_last_6mths",
    "mths_since_last_delinq", "mths_since_last_record",
    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "mths_since_last_major_derog", "acc_now_delinq", "tot_coll_amt",
    "tot_cur_bal", "total_rev_hi_lim", "acc_open_past_24mths",
    "avg_cur_bal", "bc_open_to_buy", "bc_util",
    "chargeoff_within_12_mths", "delinq_amnt",
    "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op",
    "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mort_acc",
    "mths_since_recent_bc", "mths_since_recent_bc_dlq",
    "mths_since_recent_inq", "mths_since_recent_revol_delinq",
    "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl",
    "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl",
    "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats",
    "num_tl_120dpd_2m", "num_tl_30dpd", "num_tl_90g_dpd_24m",
    "num_tl_op_past_12m", "pct_tl_nvr_dlq", "percent_bc_gt_75",
    "pub_rec_bankruptcies", "tax_liens", "tot_hi_cred_lim",
    "total_bal_ex_mort", "total_bc_limit", "total_il_high_credit_limit",
]

model_vars = [
    col for col in candidate_vars
    if col in accepts_univariate_analysis.columns
]

accepts_univariate_analysis = accepts_univariate_analysis[
    model_vars + ["issue_month", "bad"]
].copy()

In [ ]:
# apply transformations

accepts_univariate_analysis = add_model_candidate_features(
    accepts_univariate_analysis
)

In [ ]:
# plot separation charts for the candidate variables

plot_cols = accepts_univariate_analysis.drop(columns="bad").columns.tolist()

plot_separation_grid(
    accepts_univariate_analysis,
    cols=plot_cols,
    bins=5,
    ncols=4
)

### Univariate separation summary

The separation charts are mostly in line with expectations with strong univariate signals from FICO, credit history, recent credit activity and credit limit utilisation.

#### Top individual variables

* `fico_avg`: This is one of the strongest and cleanest predictors. Bad rates fall materially as FICO increases, with a clear monotonic relationship. This should be a core modelling variable

* `account_age_mths`: Longer credit history is associated with lower bad rates. The relationship is intuitive and it should provide useful additional signal beyond FICO

* `revol_util`: Higher revolving utilisation is strongly associated with higher bad rates. This is a key affordability/credit pressure indicator

* `bc_util`: Bankcard utilisation shows a similarly strong positive relationship with bad outcome. It captures more specific revolving credit pressure and is likely to be useful alongside or instead of `revol_util`

* `inq_last_6mths`: Borrowers with recent credit inquiries have materially higher bad rates. This is a strong recent-credit-seeking signal and should be considered for the model

#### Other useful variables

* Recent account-opening variables such as `acc_open_past_24mths`, `num_tl_op_past_12m`, `mo_sin_rcnt_tl`, `mo_sin_rcnt_rev_tl_op` and `mths_since_recent_inq` show useful separation. These suggest that recent credit activity is associated with higher risk which is in line with intuition

* Available credit and limit variables such as `bc_open_to_buy`, `total_bc_limit`, `total_rev_hi_lim` and `tot_hi_cred_lim` generally show lower bad rates for borrowers with more available or established credit capacity

* Account depth and mix variables such as `mort_acc`, `mo_sin_old_rev_tl_op`, `mo_sin_old_il_acct`, `num_bc_tl`, `num_il_tl`, `num_op_rev_tl` and `total_acc` show some useful signal, although the relationships are often non-linear

* Delinquency history variables such as `mths_since_last_delinq`, `mths_since_last_major_derog`, `mths_since_recent_bc_dlq` and `mths_since_recent_revol_delinq` show risk variation, but the patterns are less smooth. Missing values should be handled carefully because they may indicate no prior adverse event

* Application variables also add signal. `home_ownership` shows higher bad rates for renters compared with mortgage holders, while `purpose` shows meaningful variation across loan purposes

#### Sparse / one-bin variables

* Several adverse-event variables produce only one meaningful bin, including `acc_now_delinq`, `tot_coll_amt`, `pub_rec`, `chargeoff_within_12_mths`, `delinq_amnt`, `num_tl_120dpd_2m`, `num_tl_30dpd`, `num_tl_90g_dpd_24m`, `pub_rec_bankruptcies` and `tax_liens`

* This is because these variables are highly sparse and positively skewed: most borrowers have no recorded adverse event. Quantile binning therefore cannot create five distinct bins

* These variables should not be dismissed automatically. They are conceptually important adverse credit markers, but they may be better treated as binary flags, for example `0` versus `1+`, rather than continuous binned variables

Based on the univariate analysis:
- I will enter FICO, credit age, utilisation, recent inquiry/activity, available credit and selected account-mix variables as they are for modelling.
- sparse adverse-event variables will be transformed into flags
- all variables will be checked for stability using PSI in the model validation phase

In [ ]:
# qa - check distributions of 'skewed' variables

sparse_adverse_vars = [
    "acc_now_delinq",
    "tot_coll_amt",
    "pub_rec",
    "chargeoff_within_12_mths",
    "delinq_amnt",
    "num_tl_120dpd_2m",
    "num_tl_30dpd",
    "num_tl_90g_dpd_24m",
    "pub_rec_bankruptcies",
    "tax_liens",
]

for col in sparse_adverse_vars:
    print(col)
    print(accepts_univariate_analysis[col].value_counts(dropna=False).head(10))
    print()

## 5. Assess how historical credit risk policies shaped portfolio composition

The dataset does not contain policy decision flags / log or explicit underwriting rules. I am approching this as an inference problem, using the observed portfolio characteristic to infer how risk, apetite, pricing strategy and product mix may have changed over time.

The aim is not to identify specific policy changes with certainty, but to assess whether the accepted portfolio shows evidence of expansion, tightening or shifts in borrower risk profile.

I am focusing the analysis on four areas:

1. **Direct policy flag check**
   - Review the distribution of `policy_code` over time
   - Assess whether it provides useful evidence of distinct policy populations

2. **Portfolio risk appetite over time**
   - Review changes in accepted borrower risk profile using:
     - grade / sub-grade mix
     - FICO distribution
     - DTI (debt-to-income) distribution

3. **Product and pricing strategy over time**
   - Review changes in:
     - term mix
     - loan amount mix
     - interest rate by grade
     - loan purpose mix

4. **Accepted versus rejected applications**
   - Compare accepted and rejected application volumes over time as a proxy for changes in acceptance appetite
   - Review differences in common application characteristics where comparable fields are available

The section concludes by summarising whether the evidence suggests portfolio expansion, tightening, or changes in product and pricing strategy, and how these shifts have shaped the current accepted portfolio.

In [ ]:
# copy the full df and calculate avg fico score

policy_df = accepts_full.copy()

policy_df["fico_avg"] = (
    policy_df["fico_range_low"] + policy_df["fico_range_high"]
) / 2


Direct policy flag check

In [ ]:
policy_code_mix = pct_mix(policy_df, "issue_year", "policy_code")
policy_code_mix

Portfolio risk appetite over time

In [ ]:
plot_mix(policy_df, "grade", "Grade Mix by Issue Year")

In [ ]:
plot_yearly_metric(policy_df, "fico_avg", "Median FICO by Issue Year")

In [ ]:
plot_yearly_metric(policy_df, "dti", "Median DTI by Issue Year")

In [ ]:
plot_yearly_metric(policy_df, "loan_amnt", "Median Loan Amount by Issue Year")

In [ ]:
yearly_summary(
    policy_df,
    cols=["fico_avg", "dti", "loan_amnt"]
)

Product and pricing strategy over time

In [ ]:
plot_mix(policy_df, "term", "Term Mix by Issue Year")

In [ ]:
plot_mix(policy_df, "purpose", "Loan Purpose Mix by Issue Year")

In [ ]:
plot_interest_rate_by_grade(policy_df)


Accepted versus rejected application volumes

In [ ]:
# delete the previous datasets

del accepts_full
del accepts_univariate_analysis

gc.collect()

In [ ]:
# rejected applications: read only application date in chunks

rejected_volume = count_applications_by_year(
    paths.rejects,
    date_col="Application Date",
    output_col="rejected",
)

rejected_volume


In [ ]:
# accepted applications: read only issue date in chunks

accepted_volume = count_applications_by_year(
    paths.accepts,
    date_col="issue_d",
    output_col="accepted",
    date_format="%b-%Y",
)

accepted_volume

In [ ]:
application_volume = build_application_volume_summary(
    accepted_volume,
    rejected_volume,
)

application_volume

In [ ]:
# visualise accepted / rejected volume and proxy acceptance rate

plot_application_volume_and_acceptance(application_volume)

### Key findings: how historical credit risk policies shaped portfolio composition

* The accepted portfolio changed materially over time. Although the dataset does not contain explicit policy rules, changes in grade mix, borrower profile, pricing, product terms and application volumes suggest shifts in risk appetite and portfolio strategy

* The portfolio broadened from a more prime-focused mix into a wider risk spectrum. Earlier years had a larger share of A and B grade loans, while later years show greater exposure to B, C and D grades

* The introduction and growth of 60-month loans from around 2010 changed the product mix. This increased exposure to longer-term lending, which generally appears to carry higher credit risk than 36-month loans

* Borrower risk profile weakened during the expansion period. Median FICO declined between 2011 and 2015, while median DTI increased steadily, suggesting acceptance of more leveraged borrowers

* Loan sizes increased substantially. Median loan amount rose from around 6k in 2007 to around 13k-14k in later years, increasing exposure per borrower

* Pricing became more differentiated by risk. Median interest rates generally increased over time, especially for weaker grades, indicating risk-based pricing adjustments

* Purpose mix became more concentrated. Debt consolidation and credit card refinancing became the dominant use cases, concentrating the portfolio around re-financing loans

* Application volumes increased sharply from around 2013 onwards, while the proxy acceptance rate fell in later years. This suggests demand expanded strongly, while acceptance policy became more selective

* Overall, the evidence points to portfolio expansion and risk broadening up to around 2015–2016, followed by signs of tightening or improved risk selection from 2016 onwards

## 6. Provide any additional insights that could influence credit risk decisioning

This section summarises additional portfolio insights that could support credit risk decisioning. The focus is on areas where observed risk patterns could inform policy rules, segmentation, pricing or monitoring.

### Key additional insights

* **Term is an important risk dimension.**  
  60-month loans show materially higher bad rates than 36-month loans. This suggests term should be considered in affordability assessment, pricing and/or policy cut-offs, rather than treated only as a product choice

* **Recent credit-seeking behaviour is a strong risk signal.**  
  Variables such as recent inquiries and newly opened trades show clear separation. This supports using recent credit activity as a policy or model input, for example through tighter treatment of applicants with high recent search or account-opening activity

* **Utilisation and available credit provide strong affordability signals.**  
  High revolving or bankcard utilisation is associated with higher bad rates, while higher available bankcard credit is associated with lower risk. These variables could support affordability overlays or additional review thresholds

* **Debt consolidation and credit card refinancing dominate the portfolio.**  
  The portfolio is heavily concentrated in refinancing-led purposes. This may be commercially attractive, but it also means decisioning should consider whether the loan is genuinely reducing risk or increasing total indebtedness

* **Risk appetite changed over time, so monitoring is important.**  
  Changes in FICO, DTI, grade mix, loan size and application volumes suggest that portfolio risk can shift materially as strategy changes. Ongoing vintage monitoring, population stability checks and bad-rate tracking by segment should be part of model governance

  Overall, the analysis suggests that credit risk decisioning should not rely only on a single application score. Term, recent credit activity, utilisation, adverse credit events and refinancing purpose all provide useful context for policy design, pricing and post-deployment monitoring.